# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [1]:
# !pip install nltk

In [5]:
import ssl
import urllib.request
import nltk

# SSL sertifika kontrolünü urllib (indirme) seviyesinde tamamen kapatıyoruz
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE
urllib.request.install_opener(urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx)))

# NLTK'ya bu onaysız güvenlik bağlamını kullanmasını söylüyoruz
ssl._create_default_https_context = ssl._create_unverified_context

# Ve indirmeleri başlatıyoruz...
nltk.download('stopwords')
nltk.download('punkt')       
nltk.download('punkt_tab')   
nltk.download('wordnet')
nltk.download('omw-1.4')

print("\nNihayet! Tüm NLTK paketleri başarıyla indirildi. 🎉")

[nltk_data] Downloading package stopwords to /Users/macos/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /Users/macos/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /Users/macos/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /Users/macos/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/macos/nltk_data...



Nihayet! Tüm NLTK paketleri başarıyla indirildi. 🎉


In [6]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [7]:
import string

# --- 1.1 Noktalama İşaretlerini Kaldır ---
def noktalama_kaldir(metin):
    # string.punctuation kütüphanesini kullanarak tüm noktalama işaretlerini filtreliyoruz
    return "".join([karakter for karakter in metin if karakter not in string.punctuation])

# Fonksiyonu 'text' sütununa uygulayıp 'clean_text' adında yeni bir sütun oluşturuyoruz
df['clean_text'] = df['text'].apply(noktalama_kaldir)


# --- 1.2 Küçük Harf Yap ---
def kucuk_harf_yap(metin):
    # Metni tamamen küçük harflere çeviriyoruz
    return metin.lower()

# Fonksiyonu yeni oluşturduğumuz 'clean_text' sütununa uygulayıp üzerine yazıyoruz
df['clean_text'] = df['clean_text'].apply(kucuk_harf_yap)

# Yaptığımız değişiklikleri görmek için ilk 5 satıra bakalım
df[['text', 'clean_text']].head()

,text,clean_text
0,Subject: naturally irresistible your corporate...,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",subject do not have money get software cds fr...


### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [9]:
def kucuk_harf_yap(metin):
    # Metni tamamen küçük harflere çeviriyoruz
    return metin.lower()

# Fonksiyonu 'clean_text' sütununa uyguluyoruz
df['clean_text'] = df['clean_text'].apply(kucuk_harf_yap)

# Kontrol edelim
df[['clean_text']].head()

,clean_text
0,subject naturally irresistible your corporate ...
1,subject the stock trading gunslinger fanny is...
2,subject unbelievable new homes made easy im w...
3,subject color printing special request addit...
4,subject do not have money get software cds fr...


### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [12]:
import re

def sayilari_kaldir(metin):
    # Rakamları temizliyoruz
    return re.sub(r'\d+', '', metin)

df['clean_text'] = df['clean_text'].apply(sayilari_kaldir)
# Kontrol edelim
df[['clean_text']].head()

,clean_text
0,subject naturally irresistible corporate ident...
1,subject stock trading gunslinger fanny merrill...
2,subject unbelievable new homes made easy im wa...
3,subject color printing special request additio...
4,subject money get software cds software compat...


### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [14]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# İngilizce durdurma kelimeleri listesini alalım
durma_kelimeleri = set(stopwords.words('english'))

def stopwords_kaldir(metin):
    # Kelimelere ayır (tokenization)
    kelimeler = word_tokenize(metin)
    # Gereksizleri at
    temiz_kelimeler = [k for k in kelimeler if k not in durma_kelimeleri]
    return " ".join(temiz_kelimeler)

df['clean_text'] = df['clean_text'].apply(stopwords_kaldir)
# Kontrol edelim
df[['clean_text']].head()

,clean_text
0,subject naturally irresistible corporate ident...
1,subject stock trading gunslinger fanny merrill...
2,subject unbelievable new homes made easy im wa...
3,subject color printing special request additio...
4,subject money get software cds software compat...


### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [15]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Lemmatizer objesini oluşturuyoruz
lemmatizer = WordNetLemmatizer()

def lemmatize_et(metin):
    # Önce metni kelimelere ayırıyoruz
    kelimeler = word_tokenize(metin)
    # Her kelimeyi lemmatize edip bir listede topluyoruz
    lem_kelimeler = [lemmatizer.lemmatize(kelime) for kelime in kelimeler]
    # Yönergede istendiği gibi dize (string) olarak geri döndürüyoruz
    return " ".join(lem_kelimeler)

# Fonksiyonu 'clean_text' sütununa uyguluyoruz
df['clean_text'] = df['clean_text'].apply(lemmatize_et)

# Sonucu kontrol edelim
df[['clean_text']].head()

,clean_text
0,subject naturally irresistible corporate ident...
1,subject stock trading gunslinger fanny merrill...
2,subject unbelievable new home made easy im wan...
3,subject color printing special request additio...
4,subject money get software cd software compati...


## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer nesnesini oluşturuyoruz
vectorizer = CountVectorizer()

# clean_text sütunundaki metinleri Bag-of-Words matrisine dönüştürüyoruz
X_bow = vectorizer.fit_transform(df['clean_text'])

# Hedef değişkenimizi (spam sütunu) belirliyoruz
y = df['spam']

print(f"Sözlük boyutu: {len(vectorizer.get_feature_names_out())} benzersiz kelime.")
print(f"Matris boyutu: {X_bow.shape}")

Sözlük boyutu: 30988 benzersiz kelime.
Matris boyutu: (5728, 30988)


### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

# Naive Bayes modelimizi oluşturuyoruz
nb_model = MultinomialNB()

# Modeli çapraz doğrulama ile test ediyoruz (cv=5 veriyi 5 parçaya böler)
skorlar = cross_val_score(nb_model, X_bow, y, cv=5)

# Ortalama doğruluğu yazdıralım
print(f"Çapraz Doğrulama Skorları: {skorlar}")
print(f"Ortalama Model Doğruluğu: % {skorlar.mean() * 100:.2f}")

Çapraz Doğrulama Skorları: [0.98691099 0.9895288  0.991274   0.98777293 0.99213974]
Ortalama Model Doğruluğu: % 98.95


🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!